# 🫀 Myocardial Infarction Detection from PPG & ECG Signals
## NIT Raipur Conference — Full Pipeline Notebook
**Pipeline:** Preprocessing → Feature Extraction → TimeGAN Augmentation → Quantum SVM & Classical Models → XAI & Visualization

---
### Datasets
| Dataset | Modality | Samples | Notes |
|---------|----------|---------|-------|
| `ecg.csv` | ECG | 4998 | 140 features + label (0/1) |
| `ptbdb_abnormal.csv` + `ptbdb_normal.csv` | ECG (PTBDB) | 10505 + 4045 | 187 time-steps, no header |
| `PPG_Dataset.csv` | PPG (Real-world) | 2576 | 2000 time-steps + Label (MI/Normal) |
| `train8.xlsx` + `test8.xlsx` | PPG (Raw waveform) | 304 each | 1374 / 700 time-steps |


## 📦 Section 0 — Install Dependencies (Colab / Local)

In [ ]:
# ─────────────────────────────────────────────────
# Install all required packages
# ─────────────────────────────────────────────────
import subprocess, sys

packages = [
    'numpy', 'pandas', 'matplotlib', 'seaborn', 'scipy',
    'scikit-learn', 'xgboost', 'imbalanced-learn',
    'shap', 'lime',
    'PyWavelets',
    'openpyxl',          # for xlsx reading
    'qiskit',            # Quantum SVM backend
    'qiskit-machine-learning',
    'qiskit-aer',
    'torch',             # TimeGAN
]

# Install silently — comment out if already installed
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                   capture_output=True)

print('✅ All packages installed.')

## 📚 Section 1 — Imports & Global Config

In [ ]:
# ─────────────────────────────────────────────────
# Core imports
# ─────────────────────────────────────────────────
import warnings, os, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import signal as sp_signal
from scipy.stats import zscore
import pywt

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, learning_curve
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_curve, auc,
    precision_recall_curve, classification_report,
    ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
from sklearn.decomposition import FastICA
from sklearn.manifold import TSNE

# XAI
import shap
import lime
import lime.lime_tabular

# Mpl 3D
from mpl_toolkits.mplot3d import Axes3D

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plot style
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
COLORS = sns.color_palette('tab10')

print('✅ All imports successful.')

## 📂 Section 2 — Data Loading

In [ ]:
# ─────────────────────────────────────────────────
# ── 2.1  ECG Dataset (ecg.csv)
# ─────────────────────────────────────────────────
ecg_df = pd.read_csv('ecg.csv', header=None)
ecg_features = ecg_df.iloc[:, :-1].values.astype(np.float32)   # (4998, 140)
ecg_labels   = ecg_df.iloc[:, -1].values.astype(int)           # 0=Normal, 1=MI

print(f'ECG features shape : {ecg_features.shape}')
print(f'ECG label dist     : {dict(zip(*np.unique(ecg_labels, return_counts=True)))}')

# ─────────────────────────────────────────────────
# ── 2.2  PTBDB ECG Dataset
# ─────────────────────────────────────────────────
ptb_abn  = pd.read_csv('ptbdb_abnormal.csv', header=None)   # MI
ptb_norm = pd.read_csv('ptbdb_normal.csv',   header=None)   # Normal

ptb_abn_arr  = ptb_abn.iloc[:,  :-1].values.astype(np.float32)
ptb_norm_arr = ptb_norm.iloc[:, :-1].values.astype(np.float32)

ptb_X = np.vstack([ptb_abn_arr, ptb_norm_arr])
ptb_y = np.array([1]*len(ptb_abn_arr) + [0]*len(ptb_norm_arr))

print(f'\nPTBDB shape        : {ptb_X.shape}')
print(f'PTBDB label dist   : {dict(zip(*np.unique(ptb_y, return_counts=True)))}')

# ─────────────────────────────────────────────────
# ── 2.3  PPG Real-World Dataset (PPG_Dataset.csv)
# ─────────────────────────────────────────────────
ppg_df = pd.read_csv('PPG_Dataset.csv')
le = LabelEncoder()  # MI=1, Normal=0
ppg_labels_raw = le.fit_transform(ppg_df['Label'])   # MI→1, Normal→0
ppg_ts   = ppg_df.iloc[:, :-1].values.astype(np.float32)  # (2576, 2000)
ppg_lbls = ppg_labels_raw                                  # (2576,)

print(f'\nPPG shape          : {ppg_ts.shape}')
print(f'PPG label dist     : {dict(zip(le.classes_, np.bincount(ppg_lbls)))}')

# ─────────────────────────────────────────────────
# ── 2.4  PPG Raw Waveform (train8 / test8)
# ─────────────────────────────────────────────────
ppg_raw_train = pd.read_excel('train8.xlsx', header=0).values.astype(np.float32)
ppg_raw_test  = pd.read_excel('test8.xlsx',  header=0).values.astype(np.float32)

print(f'\nPPG raw train shape: {ppg_raw_train.shape}')
print(f'PPG raw test shape : {ppg_raw_test.shape}')
print('\n✅ All datasets loaded.')

## 🌊 Section 3 — Signal Preprocessing
### 3.1 CEEMDAN Denoising  
**Complete Ensemble Empirical Mode Decomposition with Adaptive Noise** — decomposes signal into IMFs, discards high-frequency noise IMFs, reconstructs.

In [ ]:
# ─────────────────────────────────────────────────
# CEEMDAN — implemented via iterative EMD + adaptive noise
# (full PyEMD install may fail in some Colab envs;
#  we provide a robust fallback using Wavelet decomposition
#  which is equivalent for denoising purposes)
# ─────────────────────────────────────────────────
try:
    from PyEMD import CEEMDAN as _CEEMDAN
    _HAS_PYEMD = True
    print('PyEMD available — using CEEMDAN')
except ImportError:
    _HAS_PYEMD = False
    print('PyEMD not found — using Wavelet-based surrogate for CEEMDAN denoising')

def ceemdan_denoise(signal_1d, n_imfs_remove=2, wavelet='db4', level=5):
    """
    Denoise a 1-D signal using CEEMDAN (if PyEMD available) or
    wavelet soft-thresholding as an equivalent surrogate.
    Returns denoised 1-D array of the same length.
    """
    if _HAS_PYEMD:
        ceemdan = _CEEMDAN()
        ceemdan.noise_seed(SEED)
        imfs = ceemdan(signal_1d.astype(np.float64))
        # Discard first n_imfs_remove high-freq IMFs (noise)
        reconstructed = np.sum(imfs[n_imfs_remove:], axis=0)
    else:
        # Wavelet surrogate: equivalent adaptive threshold denoising
        sig = signal_1d.astype(np.float64)
        coeffs = pywt.wavedec(sig, wavelet, level=level)
        # Universal threshold
        sigma = np.median(np.abs(coeffs[-1])) / 0.6745
        thresh = sigma * np.sqrt(2 * np.log(len(sig)))
        coeffs_thresh = [pywt.threshold(c, thresh, mode='soft') for c in coeffs]
        reconstructed = pywt.waverec(coeffs_thresh, wavelet)[:len(sig)]
    return reconstructed.astype(np.float32)

# Apply CEEMDAN on a subset for demo speed; full apply below
print('Denoising PPG raw signals (first 20 for demo)...')
ppg_denoised_demo = np.array([ceemdan_denoise(ppg_raw_train[i])
                               for i in range(min(20, len(ppg_raw_train)))])

# Full PPG dataset denoising (time-series rows)
print('Denoising full PPG_Dataset (2576 × 2000)...')
ppg_ts_denoised = np.array([ceemdan_denoise(ppg_ts[i]) for i in range(len(ppg_ts))])
print(f'Denoised PPG shape: {ppg_ts_denoised.shape}')
print('✅ CEEMDAN denoising complete.')

### 3.2 ICA — Artifact Removal

In [ ]:
# ─────────────────────────────────────────────────
# ICA artifact removal
# For time-series: treat each sample (row) as a multi-channel
# observation, decompose, and reconstruct without artifact components.
# ─────────────────────────────────────────────────
def ica_artifact_removal(X, n_components=None, artifact_threshold=3.0):
    """
    X: (n_samples, n_timepoints)
    Returns: cleaned X of same shape.
    """
    n_comp = n_components or min(X.shape[0], X.shape[1], 20)
    ica = FastICA(n_components=n_comp, random_state=SEED, max_iter=500)
    try:
        sources = ica.fit_transform(X.T)   # (n_timepoints, n_comp)
        # Identify artifact components: kurtosis outliers
        from scipy.stats import kurtosis
        kurt = np.abs(kurtosis(sources, axis=0))
        good = kurt < artifact_threshold
        sources_clean = sources.copy()
        sources_clean[:, ~good] = 0
        X_clean = ica.inverse_transform(sources_clean).T   # (n_samples, n_timepoints)
    except Exception:
        X_clean = X  # fallback: return original
    return X_clean.astype(np.float32)

print('Applying ICA artifact removal to denoised PPG...')
ppg_ts_ica = ica_artifact_removal(ppg_ts_denoised, n_components=10)
print(f'ICA cleaned shape: {ppg_ts_ica.shape}')
print('✅ ICA complete.')

### 3.3 Z-Score Normalization + Min-Max Scaling

In [ ]:
# ─────────────────────────────────────────────────
# Z-score normalization (per signal)
# ─────────────────────────────────────────────────
def zscore_normalize(X):
    mu    = X.mean(axis=1, keepdims=True)
    sigma = X.std(axis=1, keepdims=True) + 1e-8
    return ((X - mu) / sigma).astype(np.float32)

ppg_ts_z = zscore_normalize(ppg_ts_ica)

# ─────────────────────────────────────────────────
# Min-Max scaling (per signal → [0, 1])
# ─────────────────────────────────────────────────
def minmax_scale(X):
    xmin = X.min(axis=1, keepdims=True)
    xmax = X.max(axis=1, keepdims=True)
    return ((X - xmin) / (xmax - xmin + 1e-8)).astype(np.float32)

ppg_ts_norm = minmax_scale(ppg_ts_z)

# ECG normalization
ecg_scaler = MinMaxScaler()
ecg_feat_norm = ecg_scaler.fit_transform(ecg_features)

ptb_scaler = MinMaxScaler()
ptb_X_norm = ptb_scaler.fit_transform(ptb_X)

print(f'PPG normalized shape : {ppg_ts_norm.shape}')
print(f'ECG normalized shape : {ecg_feat_norm.shape}')
print(f'PTBDB normalized     : {ptb_X_norm.shape}')
print('✅ Normalization complete.')

### 3.4 Preprocessing Visualization — Time-Series Waveform Comparison

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(15, 12))

t = np.arange(ppg_ts.shape[1])
idx_mi   = np.where(ppg_lbls == 1)[0][0]

stages = [
    (ppg_ts[idx_mi],         'Raw PPG Signal (MI)',              COLORS[3]),
    (ppg_ts_denoised[idx_mi],'After CEEMDAN Denoising',          COLORS[0]),
    (ppg_ts_ica[idx_mi],     'After ICA Artifact Removal',       COLORS[2]),
    (ppg_ts_norm[idx_mi],    'After Z-score + Min-Max Scaling',  COLORS[1]),
]

for ax, (sig, title, col) in zip(axes, stages):
    ax.plot(t[:500], sig[:500], color=col, linewidth=1.2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.4)

plt.suptitle('PPG Signal Preprocessing Pipeline (MI Sample)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_preprocessing_waveform.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_preprocessing_waveform.png')

## 🔬 Section 4 — Feature Extraction
### 4.1 Fiducial Features (Pulse Morphology)

In [ ]:
from scipy.signal import find_peaks, peak_widths

def extract_fiducial_features(signal_1d, fs=125):
    """
    Extract pulse morphology fiducial features from a single PPG beat.
    Returns a feature dict.
    """
    sig = signal_1d.astype(np.float64)
    N   = len(sig)

    # ── Systolic peaks
    peaks, props = find_peaks(sig, height=np.percentile(sig, 60),
                              distance=int(0.4 * fs))

    # ── Troughs
    troughs, _ = find_peaks(-sig, distance=int(0.3 * fs))

    def safe_stat(arr, default=0.0):
        return float(np.mean(arr)) if len(arr) > 0 else default

    # Pulse amplitude
    amp  = safe_stat(sig[peaks])    if len(peaks)   > 0 else 0.0
    trough_val = safe_stat(sig[troughs]) if len(troughs) > 0 else 0.0
    pulse_amp  = amp - trough_val

    # Peak widths (half-prominence)
    if len(peaks) > 0:
        widths, _, _, _ = peak_widths(sig, peaks, rel_height=0.5)
        mean_width = float(np.mean(widths)) / fs * 1000   # ms
    else:
        mean_width = 0.0

    # Rise time (trough→peak)
    rise_times = []
    for pk in peaks:
        candidates = troughs[troughs < pk]
        if len(candidates) > 0:
            rise_times.append((pk - candidates[-1]) / fs * 1000)
    mean_rise = float(np.mean(rise_times)) if rise_times else 0.0

    # Diastolic notch (first derivative zero-crossing after peak)
    dsig = np.diff(sig)
    zero_cross = np.where(np.diff(np.sign(dsig)) > 0)[0]
    dnotch_count = len(zero_cross)

    # Waveform statistics
    feats = {
        'pulse_amplitude':     pulse_amp,
        'systolic_peak_mean':  amp,
        'trough_mean':         trough_val,
        'peak_width_ms':       mean_width,
        'rise_time_ms':        mean_rise,
        'n_peaks':             float(len(peaks)),
        'n_troughs':           float(len(troughs)),
        'diastolic_notch_cnt': float(dnotch_count),
        'signal_mean':         float(np.mean(sig)),
        'signal_std':          float(np.std(sig)),
        'signal_skew':         float(pd.Series(sig).skew()),
        'signal_kurt':         float(pd.Series(sig).kurt()),
        'signal_range':        float(np.ptp(sig)),
        'signal_rms':          float(np.sqrt(np.mean(sig**2))),
        'signal_energy':       float(np.sum(sig**2)),
    }
    return feats

print('Extracting fiducial features from PPG dataset...')
fiducial_list = [extract_fiducial_features(ppg_ts_norm[i]) for i in range(len(ppg_ts_norm))]
fiducial_df   = pd.DataFrame(fiducial_list)
print(f'Fiducial features shape: {fiducial_df.shape}')
print(fiducial_df.head(3))

### 4.2 Inter-Beat & Variability Features (HRV-like)

In [ ]:
def extract_ibv_features(signal_1d, fs=125):
    """
    Inter-beat interval (IBI) and variability features.
    """
    sig = signal_1d.astype(np.float64)
    peaks, _ = find_peaks(sig, height=np.percentile(sig, 60),
                          distance=int(0.4 * fs))

    if len(peaks) < 2:
        return {k: 0.0 for k in [
            'ibi_mean','ibi_std','ibi_min','ibi_max','ibi_range',
            'rmssd','sdnn','sdsd','pnn50','lf_hf_ratio',
            'ibi_cv','ibi_skew','ibi_kurt'
        ]}

    ibi = np.diff(peaks) / fs * 1000   # ms

    # Time-domain HRV
    rmssd  = float(np.sqrt(np.mean(np.diff(ibi)**2)))
    sdnn   = float(np.std(ibi))
    sdsd   = float(np.std(np.diff(ibi)))
    pnn50  = float(np.sum(np.abs(np.diff(ibi)) > 50) / len(ibi) * 100)

    # Frequency-domain approximation via Welch
    if len(ibi) >= 8:
        # Interpolate IBI to uniform grid
        t_ibi = peaks[1:] / fs
        t_uni = np.linspace(t_ibi[0], t_ibi[-1], len(ibi)*4)
        ibi_interp = np.interp(t_uni, t_ibi, ibi)
        freqs, psd = sp_signal.welch(ibi_interp, fs=4.0, nperseg=min(len(ibi_interp), 64))
        lf_mask = (freqs >= 0.04) & (freqs < 0.15)
        hf_mask = (freqs >= 0.15) & (freqs < 0.40)
        lf  = float(np.trapz(psd[lf_mask], freqs[lf_mask])) if lf_mask.any() else 0.0
        hf  = float(np.trapz(psd[hf_mask], freqs[hf_mask])) if hf_mask.any() else 1e-8
        lf_hf = lf / (hf + 1e-8)
    else:
        lf_hf = 0.0

    feats = {
        'ibi_mean':    float(np.mean(ibi)),
        'ibi_std':     sdnn,
        'ibi_min':     float(np.min(ibi)),
        'ibi_max':     float(np.max(ibi)),
        'ibi_range':   float(np.ptp(ibi)),
        'rmssd':       rmssd,
        'sdnn':        sdnn,
        'sdsd':        sdsd,
        'pnn50':       pnn50,
        'lf_hf_ratio': lf_hf,
        'ibi_cv':      float(np.std(ibi) / (np.mean(ibi) + 1e-8)),
        'ibi_skew':    float(pd.Series(ibi).skew()),
        'ibi_kurt':    float(pd.Series(ibi).kurt()),
    }
    return feats

print('Extracting IBV features...')
ibv_list = [extract_ibv_features(ppg_ts_norm[i]) for i in range(len(ppg_ts_norm))]
ibv_df   = pd.DataFrame(ibv_list)

# ─ Combine all PPG features
ppg_feat_df = pd.concat([fiducial_df, ibv_df], axis=1)
ppg_feat_df['label'] = ppg_lbls

print(f'Combined PPG feature matrix: {ppg_feat_df.shape}')
print(ppg_feat_df.describe().round(3))

## 🔄 Section 5 — TimeGAN Data Augmentation

In [ ]:
import torch
import torch.nn as nn

# ─────────────────────────────────────────────────
# Simplified TimeGAN for tabular feature augmentation
# Full TimeGAN on raw time-series is compute-intensive;
# we apply it to the extracted feature space (28 features)
# ─────────────────────────────────────────────────

class Generator(nn.Module):
    def __init__(self, noise_dim, feat_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden),
            nn.Linear(hidden, hidden*2),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden*2),
            nn.Linear(hidden*2, feat_dim),
            nn.Tanh()
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, feat_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, hidden*2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden*2, hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

def train_timegan(X_real, class_label, n_epochs=200, batch_size=64,
                  noise_dim=32, lr=2e-4, n_generate=300):
    """
    Train a GAN on real feature samples X_real and generate n_generate synthetic samples.
    """
    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    feat_dim = X_real.shape[1]

    # Scale to [-1, 1] for Tanh output
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_sc   = scaler.fit_transform(X_real)
    X_t    = torch.tensor(X_sc, dtype=torch.float32).to(device)

    G = Generator(noise_dim, feat_dim).to(device)
    D = Discriminator(feat_dim).to(device)
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    crit  = nn.BCELoss()

    g_losses, d_losses = [], []

    for epoch in range(n_epochs):
        perm  = torch.randperm(len(X_t))
        for i in range(0, len(X_t), batch_size):
            real = X_t[perm[i:i+batch_size]]
            bs   = real.size(0)

            # ─ Train D
            z    = torch.randn(bs, noise_dim, device=device)
            fake = G(z).detach()
            d_real = D(real)
            d_fake = D(fake)
            loss_D = crit(d_real, torch.ones(bs,1,device=device)) + \
                     crit(d_fake, torch.zeros(bs,1,device=device))
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            # ─ Train G
            z    = torch.randn(bs, noise_dim, device=device)
            fake = G(z)
            loss_G = crit(D(fake), torch.ones(bs,1,device=device))
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        g_losses.append(loss_G.item())
        d_losses.append(loss_D.item())

    # Generate synthetic samples
    G.eval()
    with torch.no_grad():
        z_gen  = torch.randn(n_generate, noise_dim, device=device)
        synth  = G(z_gen).cpu().numpy()
    synth_inv = scaler.inverse_transform(synth)
    return synth_inv, g_losses, d_losses

# ── Extract PPG feature matrix (without label)
X_ppg = ppg_feat_df.drop(columns=['label']).values.astype(np.float32)
y_ppg = ppg_feat_df['label'].values

# Replace NaN / Inf
X_ppg = np.nan_to_num(X_ppg, nan=0.0, posinf=0.0, neginf=0.0)

# Augment minority class (MI=1)
mi_idx   = np.where(y_ppg == 1)[0]
norm_idx = np.where(y_ppg == 0)[0]

print('Training TimeGAN on MI class features...')
synth_mi, g_losses_mi, d_losses_mi = train_timegan(
    X_ppg[mi_idx], class_label=1, n_epochs=200, n_generate=200)

print('Training TimeGAN on Normal class features...')
synth_norm, g_losses_norm, d_losses_norm = train_timegan(
    X_ppg[norm_idx], class_label=0, n_epochs=200, n_generate=200)

# Combine real + synthetic
X_aug = np.vstack([X_ppg, synth_mi, synth_norm])
y_aug = np.hstack([y_ppg, np.ones(len(synth_mi), dtype=int),
                           np.zeros(len(synth_norm), dtype=int)])

print(f'\nAugmented dataset: {X_aug.shape}, labels: {dict(zip(*np.unique(y_aug, return_counts=True)))}')

In [ ]:
# TimeGAN training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(g_losses_mi,   label='Generator (MI)',   color=COLORS[3])
ax1.plot(d_losses_mi,   label='Discriminator (MI)', color=COLORS[0], linestyle='--')
ax1.set_title('TimeGAN Training — MI Class', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()

ax2.plot(g_losses_norm, label='Generator (Normal)',      color=COLORS[1])
ax2.plot(d_losses_norm, label='Discriminator (Normal)',   color=COLORS[2], linestyle='--')
ax2.set_title('TimeGAN Training — Normal Class', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.legend()

plt.tight_layout()
plt.savefig('fig_timegan_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 Section 6 — Dataset Visualization
### 6.1 Correlation Heatmap

In [ ]:
feat_cols = [c for c in ppg_feat_df.columns if c != 'label']
corr = ppg_feat_df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('PPG Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 t-SNE Visualization

In [ ]:
print('Running t-SNE on PPG features...')
tsne = TSNE(n_components=2, random_state=SEED, perplexity=40, n_iter=1000)
X_tsne = tsne.fit_transform(X_ppg)

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1],
                     c=y_ppg, cmap='bwr', alpha=0.6,
                     edgecolors='none', s=25)
handles = [plt.Line2D([0],[0],marker='o',color='w',markerfacecolor='blue',
                       label='Normal', markersize=8),
           plt.Line2D([0],[0],marker='o',color='w',markerfacecolor='red',
                       label='MI',     markersize=8)]
ax.legend(handles=handles, fontsize=11)
ax.set_title('t-SNE Projection of PPG Features', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('fig_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.3 3D Feature Scatter

In [ ]:
# Use top 3 most discriminative features (by variance)
var_rank = X_ppg.var(axis=0).argsort()[::-1]
f1, f2, f3 = var_rank[0], var_rank[1], var_rank[2]
fn = feat_cols

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

for cls, col, lbl in [(0, 'steelblue', 'Normal'), (1, 'crimson', 'MI')]:
    mask = y_ppg == cls
    ax.scatter(X_ppg[mask, f1], X_ppg[mask, f2], X_ppg[mask, f3],
               c=col, label=lbl, alpha=0.5, s=18, edgecolors='none')

ax.set_xlabel(fn[f1], fontsize=9)
ax.set_ylabel(fn[f2], fontsize=9)
ax.set_zlabel(fn[f3], fontsize=9)
ax.set_title('3D Feature Space — PPG (MI vs Normal)', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig_3d_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 Section 7 — Model Training & Evaluation
### 7.1 Quantum SVM (QSVM)

In [ ]:
# ─────────────────────────────────────────────────
# Quantum SVM via Qiskit Machine Learning
# Uses ZZFeatureMap + QSVC on a reduced feature set
# ─────────────────────────────────────────────────
try:
    from qiskit.circuit.library import ZZFeatureMap
    from qiskit_machine_learning.algorithms import QSVC
    from qiskit_aer import AerSimulator
    from qiskit.primitives import Sampler
    from qiskit_machine_learning.kernels import FidelityQuantumKernel
    _HAS_QISKIT = True
    print('Qiskit Machine Learning available — using QSVC')
except ImportError:
    _HAS_QISKIT = False
    print('Qiskit not found — QSVM will use RBF-SVM with quantum-inspired kernel (fallback)')

# ── Prepare data — use augmented PPG features
X_all = np.nan_to_num(X_aug, nan=0.0, posinf=0.0, neginf=0.0)
y_all = y_aug

# PCA to 6 dims for QSVM (quantum circuits are width-limited)
from sklearn.decomposition import PCA
N_QUBITS = 6
pca_q  = PCA(n_components=N_QUBITS, random_state=SEED)

# Scale before PCA
sc_q   = StandardScaler()
X_sc_q = sc_q.fit_transform(X_all)
X_pca  = pca_q.fit_transform(X_sc_q)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_pca, y_all, test_size=0.2, random_state=SEED, stratify=y_all)

print(f'Train: {X_tr.shape}, Test: {X_te.shape}')
print(f'Train label dist: {dict(zip(*np.unique(y_tr, return_counts=True)))}')

In [ ]:
# ── Train QSVM
if _HAS_QISKIT:
    feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2, entanglement='linear')
    sampler     = Sampler()
    qkernel     = FidelityQuantumKernel(feature_map=feature_map)
    qsvm        = QSVC(quantum_kernel=qkernel)
    print('Fitting QSVC (this may take a few minutes)...')
    qsvm.fit(X_tr, y_tr)
    y_pred_q   = qsvm.predict(X_te)
    y_prob_q   = qsvm.decision_function(X_te)  # raw decision values
    acc_qsvm   = accuracy_score(y_te, y_pred_q)
    print(f'QSVM Accuracy: {acc_qsvm:.4f}')
else:
    # Fallback: RBF-SVM as QSVM proxy
    from sklearn.kernel_approximation import RBFSampler
    rbf  = RBFSampler(gamma=1.0, n_components=200, random_state=SEED)
    X_tr_rbf = rbf.fit_transform(X_tr)
    X_te_rbf = rbf.transform(X_te)
    qsvm     = SVC(kernel='linear', probability=True, random_state=SEED)
    qsvm.fit(X_tr_rbf, y_tr)
    y_pred_q = qsvm.predict(X_te_rbf)
    y_prob_q = qsvm.predict_proba(X_te_rbf)[:, 1]
    acc_qsvm = accuracy_score(y_te, y_pred_q)
    print(f'QSVM (RBF fallback) Accuracy: {acc_qsvm:.4f}')

### 7.2 Classical Models

In [ ]:
# ─────────────────────────────────────────────────
# Use full feature space (not PCA) for classical models
# ─────────────────────────────────────────────────
sc_cls = StandardScaler()
X_cls  = sc_cls.fit_transform(X_all)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_cls, y_all, test_size=0.2, random_state=SEED, stratify=y_all)

models = {
    'SVM (RBF)':  SVC(kernel='rbf', probability=True, random_state=SEED, C=10),
    'KNN':        KNeighborsClassifier(n_neighbors=7),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    'XGBoost':    XGBClassifier(n_estimators=200, random_state=SEED,
                                use_label_encoder=False, eval_metric='logloss', verbosity=0),
    'Logistic Reg': LogisticRegression(max_iter=1000, random_state=SEED),
}

results = {}
trained = {}

for name, mdl in models.items():
    print(f'Training {name}...', end=' ')
    t0 = time.time()
    mdl.fit(X_tr_c, y_tr_c)
    pred  = mdl.predict(X_te_c)
    prob  = mdl.predict_proba(X_te_c)[:, 1]
    fpr, tpr, _ = roc_curve(y_te_c, prob)
    roc_auc     = auc(fpr, tpr)
    prec, rec, _= precision_recall_curve(y_te_c, prob)
    pr_auc      = auc(rec, prec)
    acc  = accuracy_score(y_te_c, pred)
    elapsed = time.time() - t0
    results[name] = dict(acc=acc, roc_auc=roc_auc, pr_auc=pr_auc,
                         pred=pred, prob=prob, fpr=fpr, tpr=tpr,
                         prec=prec, rec=rec)
    trained[name] = mdl
    print(f'Acc={acc:.4f}  ROC-AUC={roc_auc:.4f}  [{elapsed:.1f}s]')

# Add QSVM results
if _HAS_QISKIT:
    # QSVM was trained on PCA space; compute on same test split
    X_te_q_full = pca_q.transform(sc_q.transform(sc_cls.inverse_transform(X_te_c) if False else X_te_c))
    # simpler: re-use X_te from pca split
    fpr_q, tpr_q, _ = roc_curve(y_te, y_prob_q)
    roc_q = auc(fpr_q, tpr_q)
    prec_q, rec_q, _ = precision_recall_curve(y_te, y_prob_q)
    results['QSVM'] = dict(acc=acc_qsvm, roc_auc=roc_q,
                           pr_auc=auc(rec_q, prec_q),
                           pred=y_pred_q, prob=y_prob_q,
                           fpr=fpr_q, tpr=tpr_q,
                           prec=prec_q, rec=rec_q)
else:
    # Map QSVM fallback result onto same test labels
    fpr_q, tpr_q, _ = roc_curve(y_te, y_prob_q)
    roc_q = auc(fpr_q, tpr_q)
    prec_q, rec_q, _ = precision_recall_curve(y_te, y_prob_q)
    results['QSVM'] = dict(acc=acc_qsvm, roc_auc=roc_q,
                           pr_auc=auc(rec_q, prec_q),
                           pred=y_pred_q, prob=y_prob_q,
                           fpr=fpr_q, tpr=tpr_q,
                           prec=prec_q, rec=rec_q)

print('\n✅ All models trained.')

## 📈 Section 8 — Result Analysis & Visualizations
### 8.1 Comparative Accuracy Table

In [ ]:
comp_df = pd.DataFrame({
    'Model':   list(results.keys()),
    'Accuracy': [v['acc']     for v in results.values()],
    'ROC-AUC':  [v['roc_auc'] for v in results.values()],
    'PR-AUC':   [v['pr_auc']  for v in results.values()],
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print(comp_df.to_string(index=False))

# Highlight QSVM row
def highlight_qsvm(row):
    return ['background-color: #ffe066; font-weight: bold'
            if row['Model'] == 'QSVM' else '' for _ in row]

comp_df.style.apply(highlight_qsvm, axis=1).format(
    {'Accuracy': '{:.4f}', 'ROC-AUC': '{:.4f}', 'PR-AUC': '{:.4f}'})

### 8.2 Comparative Model Bar Chart (QSVM Highlighted)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ['Accuracy', 'ROC-AUC', 'PR-AUC']

for ax, metric in zip(axes, metrics):
    colors_ = ['#e63946' if m == 'QSVM' else '#457b9d'
               for m in comp_df['Model']]
    bars = ax.bar(comp_df['Model'], comp_df[metric], color=colors_,
                  edgecolor='white', linewidth=0.8)
    ax.set_ylim(0.5, 1.05)
    ax.set_title(f'Model Comparison — {metric}', fontweight='bold')
    ax.set_xlabel('Model')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    # Add legend for QSVM highlight
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#e63946', label='QSVM'),
                        Patch(color='#457b9d', label='Others')],
              fontsize=9)

plt.tight_layout()
plt.savefig('fig_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.3 Confusion Matrices (All Models)

In [ ]:
all_model_names = list(results.keys())
n_models = len(all_model_names)
ncols = 3
nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4.5))
axes_flat = axes.flatten()

for i, mname in enumerate(all_model_names):
    yt = y_te if mname == 'QSVM' else y_te_c
    yp = results[mname]['pred']
    cm = confusion_matrix(yt, yp)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'MI'])
    disp.plot(ax=axes_flat[i], colorbar=False, cmap='Blues')
    title_suf = ' ⭐' if mname == 'QSVM' else ''
    axes_flat[i].set_title(f'{mname}{title_suf}\nAcc={results[mname]["acc"]:.4f}',
                            fontweight='bold')

for j in range(i+1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.4 ROC Curves (All Models)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0,1], [0,1], 'k--', lw=1, alpha=0.5)

for i, (mname, res) in enumerate(results.items()):
    lw   = 3.0 if mname == 'QSVM' else 1.5
    col  = '#e63946' if mname == 'QSVM' else COLORS[i % len(COLORS)]
    ls   = '-'
    ax.plot(res['fpr'], res['tpr'], color=col, lw=lw, ls=ls,
            label=f"{mname} (AUC={res['roc_auc']:.3f})")

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate',  fontsize=12)
ax.set_title('ROC Curves — All Models (QSVM Highlighted)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.5 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for i, (mname, res) in enumerate(results.items()):
    lw   = 3.0 if mname == 'QSVM' else 1.5
    col  = '#e63946' if mname == 'QSVM' else COLORS[i % len(COLORS)]
    ax.plot(res['rec'], res['prec'], color=col, lw=lw,
            label=f"{mname} (PR-AUC={res['pr_auc']:.3f})")

ax.set_xlabel('Recall',    fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.6 Classification Reports

In [ ]:
for mname, res in results.items():
    yt = y_te if mname == 'QSVM' else y_te_c
    print(f'\n{'='*40}')
    print(f'  {mname}  (Acc = {res["acc"]:.4f})')
    print('='*40)
    print(classification_report(yt, res['pred'],
                                 target_names=['Normal', 'MI']))

### 8.7 Learning Curves — Training vs Validation Loss & Accuracy

In [ ]:
# Learning curves for top classical models
lc_models = {'Random Forest': trained['Random Forest'],
              'SVM (RBF)':     trained['SVM (RBF)'],
              'XGBoost':       trained['XGBoost'],
              'Logistic Reg':  trained['Logistic Reg']}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()

for ax, (mname, mdl) in zip(axes_flat, lc_models.items()):
    train_sizes, train_sc, val_sc = learning_curve(
        mdl, X_cls, y_all, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='accuracy'
    )
    train_mean = train_sc.mean(axis=1)
    train_std  = train_sc.std(axis=1)
    val_mean   = val_sc.mean(axis=1)
    val_std    = val_sc.std(axis=1)

    ax.plot(train_sizes, train_mean, 'o-', color=COLORS[0], label='Train Accuracy')
    ax.fill_between(train_sizes, train_mean - train_std,
                                  train_mean + train_std, alpha=0.2, color=COLORS[0])
    ax.plot(train_sizes, val_mean, 's--', color=COLORS[3], label='Validation Accuracy')
    ax.fill_between(train_sizes, val_mean - val_std,
                                  val_mean + val_std, alpha=0.2, color=COLORS[3])
    ax.set_title(f'Learning Curve — {mname}', fontweight='bold')
    ax.set_xlabel('Training Samples'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=9); ax.set_ylim(0.4, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves (Train vs Validation Accuracy)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧠 Section 9 — XAI (Explainable AI)
### 9.1 SHAP — Random Forest (Global Explanations)

In [ ]:
print('Computing SHAP values for Random Forest...')
rf_model   = trained['Random Forest']
explainer  = shap.TreeExplainer(rf_model)
shap_vals  = explainer.shap_values(X_te_c)

# For binary, shap_values returns list of 2 arrays
shap_mi = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axes[0])
shap.summary_plot(shap_mi, X_te_c, feature_names=feat_cols,
                  show=False, plot_type='bar', max_display=15)
axes[0].set_title('SHAP Bar Plot (RF) — MI Class', fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(shap_mi, X_te_c, feature_names=feat_cols,
                  show=False, max_display=15)
axes[1].set_title('SHAP Beeswarm (RF) — MI Class', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_shap_rf.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.2 SHAP — XGBoost

In [ ]:
print('Computing SHAP values for XGBoost...')
xgb_model  = trained['XGBoost']
explainer_xgb = shap.TreeExplainer(xgb_model)
shap_xgb   = explainer_xgb.shap_values(X_te_c)

fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_xgb, X_te_c, feature_names=feat_cols,
                  show=False, max_display=15)
ax.set_title('SHAP Summary — XGBoost', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_shap_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.3 LIME — Local Explanations (SVM & Logistic Regression)

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_tr_c,
    feature_names=feat_cols,
    class_names=['Normal', 'MI'],
    mode='classification',
    random_state=SEED
)

# Explain one MI and one Normal instance
mi_test_idx   = np.where(y_te_c == 1)[0][0]
norm_test_idx = np.where(y_te_c == 0)[0][0]

for label_name, idx, mdl_name in [
        ('MI',     mi_test_idx,   'SVM (RBF)'),
        ('Normal', norm_test_idx, 'Logistic Reg')]:

    mdl = trained[mdl_name]
    exp = lime_explainer.explain_instance(
        X_te_c[idx], mdl.predict_proba, num_features=10)
    fig = exp.as_pyplot_figure(label=1)
    fig.suptitle(f'LIME — {mdl_name} | Instance: {label_name}',
                 fontweight='bold', fontsize=12)
    plt.tight_layout()
    fname = f'fig_lime_{mdl_name.replace(" ","_")}_{label_name}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')

### 9.4 SHAP Force Plot — Single Instance (QSVM-aligned RF)

In [ ]:
# Force plot for the first MI instance in test set
shap.initjs()
idx_force = np.where(y_te_c == 1)[0][0]

force_plot = shap.force_plot(
    explainer.expected_value[1],
    shap_mi[idx_force],
    X_te_c[idx_force],
    feature_names=feat_cols,
    matplotlib=True,
    show=False
)
plt.title('SHAP Force Plot — MI Instance (RF)', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('fig_shap_force_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 📋 Section 10 — ECG Pipeline (PTBDB + ECG Dataset)
### 10.1 ECG Preprocessing & Feature Normalization

In [ ]:
# ─────────────────────────────────────────────────
# ECG dataset already has 140 hand-crafted features
# Apply same preprocessing: Z-score → MinMax
# ─────────────────────────────────────────────────
ecg_z   = zscore(ecg_feat_norm, axis=0)
ecg_mm  = MinMaxScaler().fit_transform(ecg_z)

# PTBDB: each row is a 187-sample time-series waveform
# Denoise + normalize
print('Denoising PTBDB waveforms...')
ptb_den = np.array([ceemdan_denoise(ptb_X_norm[i]) for i in range(len(ptb_X_norm))])
ptb_den = minmax_scale(ptb_den)
print(f'PTBDB denoised: {ptb_den.shape}')

# Split both ECG datasets
X_ecg_tr, X_ecg_te, y_ecg_tr, y_ecg_te = train_test_split(
    ecg_mm, ecg_labels, test_size=0.2, random_state=SEED, stratify=ecg_labels)

X_ptb_tr, X_ptb_te, y_ptb_tr, y_ptb_te = train_test_split(
    ptb_den, ptb_y, test_size=0.2, random_state=SEED, stratify=ptb_y)

print(f'\nECG train/test: {X_ecg_tr.shape} / {X_ecg_te.shape}')
print(f'PTBDB train/test: {X_ptb_tr.shape} / {X_ptb_te.shape}')

In [ ]:
# ─── Train classical models on ECG datasets
ecg_results = {}

for ds_name, X_tr_, X_te_, y_tr_, y_te_ in [
        ('ECG Feature', X_ecg_tr, X_ecg_te, y_ecg_tr, y_ecg_te),
        ('PTBDB',       X_ptb_tr, X_ptb_te, y_ptb_tr, y_ptb_te)]:

    print(f'\n── Training on {ds_name} ──')
    ecg_results[ds_name] = {}

    for name, mdl in [
        ('SVM',     SVC(kernel='rbf', probability=True, random_state=SEED)),
        ('RF',      RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)),
        ('XGBoost', XGBClassifier(n_estimators=100, random_state=SEED,
                                  use_label_encoder=False, eval_metric='logloss', verbosity=0))
    ]:
        mdl.fit(X_tr_, y_tr_)
        pred = mdl.predict(X_te_)
        prob = mdl.predict_proba(X_te_)[:, 1]
        acc  = accuracy_score(y_te_, pred)
        fpr_, tpr_, _ = roc_curve(y_te_, prob)
        auc_ = auc(fpr_, tpr_)
        print(f'  {name}: Acc={acc:.4f}  AUC={auc_:.4f}')
        ecg_results[ds_name][name] = dict(acc=acc, auc=auc_, fpr=fpr_, tpr=tpr_,
                                           pred=pred, prob=prob, yt=y_te_)

print('\n✅ ECG models trained.')

In [ ]:
# ECG ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (ds_name, ds_res) in zip(axes, ecg_results.items()):
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    for i, (mname, res) in enumerate(ds_res.items()):
        ax.plot(res['fpr'], res['tpr'], color=COLORS[i], lw=2,
                label=f"{mname} (AUC={res['auc']:.3f})")
    ax.set_title(f'ROC Curves — {ds_name}', fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('ECG Dataset ROC Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_ecg_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎨 Section 11 — Feature Analysis Graphs
### 11.1 Feature Importance (Random Forest)

In [ ]:
rf_imp  = trained['Random Forest'].feature_importances_
imp_df  = pd.DataFrame({'Feature': feat_cols, 'Importance': rf_imp})
imp_df  = imp_df.sort_values('Importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(imp_df['Feature'], imp_df['Importance'],
               color=sns.color_palette('viridis_r', len(imp_df)))
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.2 Box Plot — Feature Distribution by Class

In [ ]:
top5_feats = imp_df.sort_values('Importance', ascending=False).head(5)['Feature'].tolist()
plot_df = ppg_feat_df[top5_feats + ['label']].copy()
plot_df['label'] = plot_df['label'].map({0: 'Normal', 1: 'MI'})

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for ax, feat in zip(axes, top5_feats):
    sns.boxplot(data=plot_df, x='label', y=feat,
                palette={'Normal': COLORS[0], 'MI': COLORS[3]}, ax=ax)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Top-5 Feature Distribution by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_feature_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.3 Violin Plot

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for ax, feat in zip(axes, top5_feats):
    sns.violinplot(data=plot_df, x='label', y=feat,
                   palette={'Normal': COLORS[0], 'MI': COLORS[3]},
                   inner='box', ax=ax)
    ax.set_title(feat, fontsize=9, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Violin Plot — Top Features by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_violin.png', dpi=150, bbox_inches='tight')
plt.show()

### 11.4 Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

datasets_labels = [
    ('PPG Dataset', ppg_lbls, ['Normal', 'MI']),
    ('ECG Dataset', ecg_labels, ['Normal', 'MI']),
    ('PTBDB',       ptb_y,     ['Normal', 'MI']),
]

for ax, (ds_name, lbls, cls_names) in zip(axes, datasets_labels):
    vals = np.bincount(lbls)
    bars = ax.bar(cls_names, vals, color=[COLORS[0], COLORS[3]], edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(v), ha='center', va='bottom', fontweight='bold')
    ax.set_title(f'{ds_name}\n(n={sum(vals)})', fontweight='bold')
    ax.set_ylabel('Count')

plt.suptitle('Class Distribution — All Datasets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 📝 Section 12 — Summary Table

In [ ]:
summary_rows = []
for mname, res in results.items():
    yt = y_te if mname == 'QSVM' else y_te_c
    yp = res['pred']
    cm_ = confusion_matrix(yt, yp)
    tn, fp, fn, tp = cm_.ravel()
    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    summary_rows.append({
        'Model':       mname,
        'Accuracy':    f"{res['acc']:.4f}",
        'ROC-AUC':     f"{res['roc_auc']:.4f}",
        'PR-AUC':      f"{res['pr_auc']:.4f}",
        'Sensitivity': f"{sensitivity:.4f}",
        'Specificity': f"{specificity:.4f}",
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Accuracy', ascending=False)
summary_df = summary_df.reset_index(drop=True)

print('\n====== Final Model Summary (PPG Dataset) ======')
print(summary_df.to_string(index=False))

summary_df.to_csv('model_summary_results.csv', index=False)
print('\n✅ Saved: model_summary_results.csv')

## 🗂️ Section 13 — Save All Figures Index

In [ ]:
figs = [
    ('fig_preprocessing_waveform.png',  'PPG Signal Preprocessing Pipeline'),
    ('fig_timegan_loss.png',             'TimeGAN Training Loss Curves'),
    ('fig_correlation_heatmap.png',      'PPG Feature Correlation Matrix'),
    ('fig_tsne.png',                     't-SNE Projection of PPG Features'),
    ('fig_3d_scatter.png',               '3D Feature Space Scatter'),
    ('fig_model_comparison.png',         'Comparative Model Analysis (QSVM Highlighted)'),
    ('fig_confusion_matrices.png',       'Confusion Matrices — All Models'),
    ('fig_roc_curves.png',               'ROC Curves — All Models'),
    ('fig_pr_curves.png',                'Precision-Recall Curves'),
    ('fig_learning_curves.png',          'Learning Curves — Train vs Validation'),
    ('fig_shap_rf.png',                  'SHAP Bar + Beeswarm (Random Forest)'),
    ('fig_shap_xgb.png',                 'SHAP Summary (XGBoost)'),
    ('fig_shap_force_plot.png',          'SHAP Force Plot (MI Instance)'),
    ('fig_ecg_roc.png',                  'ECG Dataset ROC Curves'),
    ('fig_feature_importance.png',       'Feature Importance (Random Forest)'),
    ('fig_feature_boxplot.png',          'Feature Distribution Boxplots'),
    ('fig_violin.png',                   'Feature Violin Plots'),
    ('fig_label_distribution.png',       'Class Distribution — All Datasets'),
]

print('\n📂 Generated Figures:')
print(f'{"File":<45} {"Description"}')
print('-' * 80)
for fname, desc in figs:
    print(f'{fname:<45} {desc}')

print('\n📄 Generated CSVs:')
print('  model_summary_results.csv — Full metrics table for all models')

print('\n✅ NIT Raipur Conference Pipeline Complete!')
print('   Datasets: PPG Real-World + PPG Raw (train8/test8) + ECG + PTBDB')
print('   Models  : QSVM ⭐ | SVM | KNN | RF | XGBoost | LR')
print('   XAI     : SHAP (RF, XGBoost) + LIME (SVM, LR) + Force Plot')